# 02. LangChain RAG 구축
**Day 5**

> Day 4 **03 RAG** 에서 직접 짠 `split_text` · `search` 를  
> LangChain **Document → Splitter → Embedding → VectorStore** 로 바꿉니다.

**데이터:** `../4일차_찐/samples/pdf_samples/학칙.pdf` (이후 03에서 전체 PDF 확장)

---
## 0. 설치

In [ ]:
!pip install PyMuPDF pypdf langchain langchain_community

---
## 1. Day 4 RAG vs LangChain RAG

| Day 4 (직접 구현) | LangChain |
|------------------|-----------|
| `pdf_to_text` | `PyMuPDFLoader` |
| `split_text` | `RecursiveCharacterTextSplitter` |
| `tokenize` + 교집합 | `OpenAIEmbeddings` + 벡터 유사도 |
| `INDEX` 리스트 | `FAISS` VectorStore |

---
## 2. Document 로드

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
import os 
# PDF 파일을 읽어서 텍스트 데이터를 추출합니다.
pdf_path = os.path.join(os.getcwd(),'samples/rag_samples/학칙.pdf')
loader = PyPDFLoader(pdf_path)

In [ ]:
loader

In [ ]:
data_nyc = loader.load()
print(data_nyc)

---
## 3. Text Split (청킹)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 텍스트 데이터를 1000자 단위로 나눕니다. overlap은 100자로 설정합니다.
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

In [ ]:
all_splits = text_splitter.split_documents(data_nyc)

In [ ]:
print(type(all_splits[0]))

In [ ]:
all_splits

In [ ]:
for i, split in enumerate(all_splits):
    print(f"Split {i+1}:------------------------------------\n")
    print(split)

In [ ]:
print(all_splits[50].page_content)
print('----------------------')
print(all_splits[51].page_content)

news.pdf 동일하게 진행

변수명:

loader_news, data_news, news_splits

In [ ]:
# 코드 작성

In [ ]:
for i, split in enumerate(news_splits):
    print(f"Split {i+1}:------------------------------------")
    print(split)

In [ ]:
all_splits.extend(news_splits)

In [ ]:
all_splits

---
## 4. Embedding 

In [ ]:
!pip install langchain_chroma langchain_openai

In [ ]:
from langchain_openai import OpenAIEmbeddings 
from dotenv import load_dotenv
import os

load_dotenv()

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

In [ ]:
embedding = OpenAIEmbeddings(model='text-embedding-3-large', api_key=OPENAI_API_KEY)

In [ ]:
v = embedding.embed_query('연세대학교에서 논문제출 시한을 연장한 학생은 휴학할 수 있는가? 예외는 무엇인가?')

In [ ]:
len(v)

#### Embedding 모델

text-embedding-3-large : 높은 정확도 (*한글)

text-embedding-3-small : 비용 대비 성능이 뛰어남

---
## 4. 벡터 저장

다양한 저장소 존재 (FAISS, **Chroma**, Qdrant, ....)

In [ ]:
from langchain_chroma import Chroma
import os

persist_directory = 'chroma_store'	

# 저장된 크로마 DB가 없다면 새로 만들기
if not os.path.exists(persist_directory):
    print("Creating new Chroma store")
    vectorstore = Chroma.from_documents(
        documents=all_splits,
        embedding=embedding,
        persist_directory=persist_directory
    )

else:
    print("Loading existing Chroma store")
    vectorstore = Chroma(		
        persist_directory=persist_directory, 
        embedding_function=embedding
    )

In [ ]:
#FAISS 예시
from langchain_community.vectorstores import FAISS

embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(all_splits, embeddings)

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k":3})

docs = retriever.invoke("연세대학교에서 논문제출 시한을 연장한 학생은 휴학할 수 있는가? 예외는 무엇인가?")

In [ ]:
docs

In [ ]:
for d in docs:
    print(d)
    print('------')

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

question = '석사학위과정 수업연한은?'
chat = ChatOpenAI(model="gpt-4o-mini")
retriever = vectorstore.as_retriever(search_kwargs={'k': 3})

In [ ]:
rag_prompt = ChatPromptTemplate.from_template("""
사용자의 질문에 대해 아래 context에 기반하여 답변하라

문맥:
{context}

질문: {question}
""")

### rag 답변 생성해보기

In [ ]:
# 힌트
def format_docs(docs):
    return '\n\n'.join(d.page_content for d in docs)

## Retriever -> Tool

In [ ]:
from langchain_core.tools import create_retriever_tool
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(model="gpt-4o-mini")
search_tool = create_retriever_tool(
    retriever,
    name = 'search_pdf_documents',
    description = ' pdf samples 안의 문서함에서 질문과 관련된 내용을 검색합니다.'
)

In [ ]:
retriever

In [ ]:
retriever.invoke('석사 수업연한')

In [ ]:
search_tool.invoke('석사 수업연한')

In [ ]:
tools = [search_tool]
tool_dict = {'search_pdf_documents':search_tool}

llm_with_tools = llm.bind_tools(tools)

messages = [
    HumanMessage('석사학위과정 수업연한은?')
]

response = llm_with_tools.invoke('석사학위과정 수업연한은?')
messages.append(response)

In [ ]:
response.tool_calls

In [ ]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]] # (7) tool_dict를 사용하여 도구 함수를 선택
    print(tool_call["args"]) # (8) 도구 호출 시 전달된 인자 출력
    tool_msg = selected_tool.invoke(tool_call) # (9) 도구 함수를 호출하여 결과를 반환
    messages.append(tool_msg)

messages

In [ ]:
llm_with_tools.invoke(messages)